# Food Vision v2 — train and evaluate ConvNeXt-Tiny

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

About 3–3.5 hours: ~5 min to fetch Food-101, ~2.5–3 h to train 12 epochs, ~5 min
to evaluate the held-out test split.

**If Colab disconnects:** reconnect and **Run all** again. Training state is
saved to Google Drive after every epoch, so it continues from the last finished
epoch instead of starting over.

Keep this tab open. When asked, allow Google Drive access.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Drive first, so the permission prompt appears now rather than hours from now.
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/food_vision_v2"
CHECKPOINT = f"{DRIVE_DIR}/food_vision_convnext_tiny.pt"
!mkdir -p {DRIVE_DIR}

In [ ]:
# Pull the repo so training uses the same model definition and preprocessing
# transform the API serves.
BRANCH = "main"
!rm -rf /content/foodVision
!git clone --depth 1 -b {BRANCH} https://github.com/vidit-16/foodVision.git /content/foodVision
%cd /content/foodVision
!pip install -q -r requirements.txt

## Dataset

torchvision fetches Food-101 and reads the official `meta/train.json` and
`meta/test.json` split files.

In [ ]:
from torchvision.datasets import Food101

train = Food101(root="/content/data", split="train", download=True)
test = Food101(root="/content/data", split="test", download=True)

print(f"train images: {len(train):,}")   # 75,750
print(f"test images:  {len(test):,}")    # 25,250
print(f"classes:      {len(train.classes)}")
assert len(train) == 75750 and len(test) == 25250

In [ ]:
# Two batches end to end. Stops Run all here if anything is miswired, instead
# of failing silently and wasting the training run.
out = !python training/train.py --data-dir /content/data --smoke-test 2>&1
print(out.n)
assert "smoke test completed" in out.n, "smoke test failed, see output above"

## Train

ConvNeXt-Tiny (ImageNet-22k pretrained), AdamW with warmup and cosine decay,
label smoothing, TrivialAugment and random erasing, 12 epochs at batch size 64.
10% of the train split is held back for validation; the test split stays closed.

Each epoch prints its time and an estimate of what's left. The best-validation
weights and the resumable state both live on Drive.

In [ ]:
import os

!python training/train.py     --data-dir /content/data     --epochs 12     --batch-size 64     --output {CHECKPOINT}     --checkpoint-dir {DRIVE_DIR}     --resume

assert os.path.exists(CHECKPOINT), "training did not produce a checkpoint, see output above"

## Evaluate

Scores the checkpoint on the 25,250 test images it has never seen, through the
same transform `app/preprocessing.py` applies at inference time.

In [ ]:
# No checksum exists for the new checkpoint yet; it is computed below.
!FOODVISION_WEIGHTS_PATH={CHECKPOINT} FOODVISION_WEIGHTS_SHA256=  python evaluation/evaluate.py --data-dir /content/data

assert os.path.exists("evaluation/results.json"), "evaluation failed, see output above"

## Summary of results

In [ ]:
import hashlib, json, pathlib

digest = hashlib.sha256(pathlib.Path(CHECKPOINT).read_bytes()).hexdigest()
results = json.load(open("evaluation/results.json"))

print("=" * 72)
print(f"SHA256: {digest}")
print(f"top-1: {results['top1']*100:.2f}%   top-5: {results['top5']*100:.2f}%")
print(f"over {results['images']:,} held-out test images")
print(f"checkpoint on Drive: MyDrive/food_vision_v2/food_vision_convnext_tiny.pt")
print("=" * 72)
print(pathlib.Path("evaluation/RESULTS.md").read_text())